# Importo librerías

In [14]:
import numpy as np
import pandas as pd
from pathlib import Path
from itertools import product
import warnings
import pickle
import re
from scipy.stats import wilcoxon

# from ydata_profiling import ProfileReport
import matplotlib.pyplot as plt

### Helper funcs

In [15]:
def sanitize_for_pickle(obj):
    if isinstance(obj, pd.DataFrame):
        obj = obj.copy()
        for col in obj.columns:
            if str(obj[col].dtype).startswith("string"):
                obj[col] = obj[col].astype(object)
        return obj

    elif isinstance(obj, pd.Series):
        if str(obj.dtype).startswith("string"):
            return obj.astype(object)
        return obj

    elif isinstance(obj, dict):
        return {k: sanitize_for_pickle(v) for k, v in obj.items()}

    elif isinstance(obj, list):
        return [sanitize_for_pickle(x) for x in obj]

    elif isinstance(obj, tuple):
        return tuple(sanitize_for_pickle(x) for x in obj)

    else:
        return obj

In [16]:
def summarize_results(all_results):
    def dataset_metadata(dataset_name):
        dataset_dir = Path("dataset") / "data_base" / dataset_name
        dat_files = sorted([p for p in dataset_dir.glob(f"{dataset_name}-{dataset_name}-cc.dat") if p.is_file()])
        if not dat_files:
            return {
                "n_instances": np.nan,
                "n_categorical_features": np.nan,
                "n_numerical_features": np.nan,
                "n_total_features": np.nan,
                "pct_numerical_features": np.nan,
                "pct_categorical_features": np.nan,
                "n_classes": np.nan,
            }

        file_path = dat_files[0]
        n_instances = np.nan
        in_data = False

        n_categorical_features = 0
        n_numerical_features = 0
        last_attribute_categories = None

        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            for raw_line in f:
                line = raw_line.strip()
                if not line or line.startswith("%"):
                    continue

                lower = line.lower()
                if lower.startswith("@attribute"):
                    if "{" in lower and "}" in lower:
                        n_categorical_features += 1
                        last_attribute_categories = line[line.find("{") + 1: line.rfind("}")].split(",")
                    else:
                        n_numerical_features += 1
                elif lower.startswith("@data"):
                    in_data = True
                elif in_data:
                    n_instances = 0 if pd.isna(n_instances) else n_instances
                    n_instances += 1

        n_categorical_features_no_class = n_categorical_features - 1
        n_total_features = n_numerical_features + n_categorical_features_no_class

        return {
            "n_instances": n_instances,
            "n_categorical_features": n_categorical_features_no_class,
            "n_numerical_features": n_numerical_features,
            "n_total_features": n_total_features,
            "pct_numerical_features": (100 * n_numerical_features / n_total_features) if n_total_features else np.nan,
            "pct_categorical_features": (100 * n_categorical_features_no_class / n_total_features) if n_total_features else np.nan,
            "n_classes": len(last_attribute_categories) if last_attribute_categories is not None else np.nan,
        }

    rows = []
    for res in all_results:
        dataset = res["dataset"]
        dataset_info = dataset_metadata(dataset)
        baseline_acc_col = "acc" if "acc" in res["baseline_df"].columns else "accuracy"
        baseline_mean = (
            res["baseline_df"][[baseline_acc_col, "bal_acc", "f1_macro", "precision_macro", "recall_macro"]]
            .mean()
            .add_prefix("baseline_")
            .rename(index={f"baseline_{baseline_acc_col}": "baseline_acc"})
        )
        class_df = res["class_summary_df"].copy()
        if "class_acc_mean" not in class_df.columns:
            if "accuracy_mean" in class_df.columns:
                class_df["class_acc_mean"] = class_df["accuracy_mean"]
            elif "acc_mean" in class_df.columns:
                class_df["class_acc_mean"] = class_df["acc_mean"]
        class_df["method_base"] = class_df["method"].str.split("_", n=1).str[0]
        best_class = (
            class_df.sort_values(
                by=["method_base", "f1_macro_mean", "bal_acc_mean", "class_acc_mean", "precision_macro_mean", "recall_macro_mean"],
                ascending=[True, False, False, False, False, False],
            )
            .groupby("method_base", as_index=False)
            .head(1)
        )
        removal_df = res["removal_summary_df"].copy()
        removal_df["method_base"] = removal_df["filter"].str.split("_", n=1).str[0]
        best_removal = (
            removal_df.merge(
                best_class[["dataset", "noise_type", "noise_pct", "seed", "k", "method", "method_base"]],
                left_on=["dataset", "noise_type", "noise_pct", "seed", "k", "method_base"],
                right_on=["dataset", "noise_type", "noise_pct", "seed", "k", "method_base"],
                how="inner",
                suffixes=("", "_bestclass"),
            )
        )
        for _, c_row in best_class.iterrows():
            m = c_row["method_base"]
            class_rows = res["classification_df"][res["classification_df"]["method"] == c_row["method"]]
            n_invalid_trainings = int(not class_rows["valid_classification"].all()) if not class_rows.empty else np.nan
            r_row = best_removal[best_removal["method_base"] == m].iloc[0] if not best_removal[best_removal["method_base"] == m].empty else None
            row = {
                "dataset": dataset,
                **dataset_info,
                "method_base": m,
                "n_invalid_trainings": n_invalid_trainings,
                "acc_diff": c_row["class_acc_mean"] - baseline_mean["baseline_acc"],
                "bal_acc_diff": c_row["bal_acc_mean"] - baseline_mean["baseline_bal_acc"],
                "f1_macro_diff": c_row["f1_macro_mean"] - baseline_mean["baseline_f1_macro"],
                "prec_macro_diff": c_row["precision_macro_mean"] - baseline_mean["baseline_precision_macro"],
                "rec_macro_diff": c_row["recall_macro_mean"] - baseline_mean["baseline_recall_macro"],
                "removal_filter": r_row["filter"] if r_row is not None else np.nan,
                "rem_f1_removal_mean": r_row["f1_removal_mean"] if r_row is not None else np.nan,
                "rem_mcc_mean": r_row["mcc_mean"] if r_row is not None else np.nan,
                "rem_recall_removal_mean": r_row["recall_removal_mean"] if r_row is not None else np.nan,
                "rem_precision_removal_mean": r_row["precision_removal_mean"] if r_row is not None else np.nan,
                "rem_acc_removal_mean": r_row["acc_removal_mean"] if r_row is not None else np.nan,
                "rem_specificity_mean": r_row["specificity_mean"] if r_row is not None else np.nan,
                "rem_removed_pct_mean": r_row["removed_pct_mean"] if r_row is not None else np.nan,
                **baseline_mean.to_dict(),
                "class_method": c_row["method"],
                "class_acc_mean": c_row["class_acc_mean"],
                "class_f1_macro_mean": c_row["f1_macro_mean"],
                "class_bal_acc_mean": c_row["bal_acc_mean"],
                "class_precision_macro_mean": c_row["precision_macro_mean"],
                "class_recall_macro_mean": c_row["recall_macro_mean"],
            }
            rows.append(row)

    df = pd.DataFrame(rows)
    preferred_cols = [
        "dataset",
        "n_instances",
        "n_categorical_features",
        "n_numerical_features",
        "n_total_features",
        "pct_numerical_features",
        "pct_categorical_features",
        "n_classes",
        "method_base",
        "n_invalid_trainings",
        "acc_diff",
        "bal_acc_diff",
        "f1_macro_diff",
        "prec_macro_diff",
        "rec_macro_diff",
        "removal_filter",
        "removal_f1_removal_mean",
        "rem_mcc_mean",
        "rem_recall_removal_mean",
        "rem_precision_removal_mean",
        "rem_acc_removal_mean",
        "rem_specificity_mean",
        "rem_removed_pct_mean",
        "baseline_acc",
        "baseline_bal_acc",
        "baseline_f1_macro",
        "baseline_precision_macro",
        "baseline_recall_macro",
        "class_method",
        "class_acc_mean",
        "class_f1_macro_mean",
        "class_bal_acc_mean",
        "class_precision_macro_mean",
        "class_recall_macro_mean",
    ]
    return df[[c for c in preferred_cols if c in df.columns] + [c for c in df.columns if c not in preferred_cols]]


In [17]:
def summary_by_method(sum_res):
    return (
        sum_res[
            ["method_base","acc_diff","bal_acc_diff","f1_macro_diff",#"prec_macro_diff",
            "rem_acc_removal_mean","rem_precision_removal_mean",
            "rem_recall_removal_mean","rem_removed_pct_mean"]
        ]
        .groupby("method_base")
        .mean()
        .reset_index()
        .rename(columns={
            "method_base": "Filtro",
            "acc_diff": "class_acc",
            "bal_acc_diff": "bal_acc",
            "f1_macro_diff": "f1_macro",
            "prec_macro_diff": "prec_macro",
            "rem_acc_removal_mean": "acc",
            "rem_precision_removal_mean": "prec",
            "rem_recall_removal_mean": "recall",
            "rem_removed_pct_mean": "pct",
        })
    )


In [18]:
def wilcoxon_to_bal_acc_dif(sum_res):
    rows = []

    for mb, g in sum_res.groupby("method_base"):
        vals = g["bal_acc_diff"].dropna()

        if len(vals) == 0 or np.allclose(vals, 0):
            pval = np.nan
        else:
            try:
                pval = wilcoxon(vals, alternative="two-sided").pvalue
            except ValueError:
                pval = np.nan

        rows.append({
            "Filtro": mb,
            "bal_acc": vals.mean(),
            "p_value": pval,
        })
    return pd.DataFrame(rows)


##

In [19]:
dataset_root = Path("dataset")
keel_datasets = sorted([p.name for p in (dataset_root / "data_base").iterdir() if p.is_dir()])
print(f"Available datasets: {keel_datasets}")

Available datasets: ['autos', 'balance', 'banana', 'car', 'cleveland', 'contraceptive', 'dermatology', 'ecoli', 'flare', 'german', 'glass', 'hayes-roth', 'heart', 'ionosphere', 'iris', 'led7digit', 'lymphography', 'magic', 'monk-2', 'newthyroid', 'nursery', 'page-blocks', 'penbased', 'phoneme', 'pima', 'ring', 'satimage', 'segment', 'shuttle', 'sonar', 'spambase', 'splice', 'thyroid', 'twonorm', 'vehicle', 'vowel', 'wdbc', 'wine', 'yeast', 'zoo']


# Prueba

In [21]:
noise_k = 5
classifier = "knn"
x = [pickle.load(open(f"./resultsEvaluation/{noise_k}/seed0{seed}/{classifier}/dists_res.pkl", "rb")) for seed in range(1,6)]

In [37]:
y = [summarize_results(x[i]) for i in range(5)]
for i in range(5):
    y[i]["seed"] = i+1*np.ones(len(y[i]))
a = pd.concat(y, axis=0).\
    filter(["dataset", "seed", "method_base", "baseline_acc", "class_acc_mean"]).\
    groupby(["dataset", "method_base"]).\
        mean()
a.head(50)

seed  baseline_acc  class_acc_mean
dataset      method_base                                    
autos        CF            3.0      0.694395        0.678831
             ENN           3.0      0.694395        0.619637
             ENNTh         3.0      0.694395        0.624516
             NCNEdit       3.0      0.694395        0.654758
car          CF            3.0      0.682433        0.782775
             ENN           3.0      0.682433        0.790187
             ENNTh         3.0      0.682433        0.743533
             NCNEdit       3.0      0.682433        0.800602
cleveland    CF            3.0      0.502339        0.641582
             ENN           3.0      0.502339        0.635593
             ENNTh         3.0      0.502339        0.578395
             NCNEdit       3.0      0.502339        0.622893
dermatology  CF            3.0      0.885469        0.947410
             ENN           3.0      0.885469        0.951354
             ENNTh         3.0      0.885469        0.934038
             NCNEdit       3.0      0.885469        0.965861
ecoli        CF            3.0      0.758990        0.841071
             ENN           3.0      0.758990        0.831580
             ENNTh         3.0      0.758990        0.820272
             NCNEdit       3.0      0.758990        0.834565
german       CF            3.0      0.678800        0.760200
             ENN           3.0      0.678800        0.765600
             ENNTh         3.0      0.678800        0.756200
             NCNEdit       3.0      0.678800        0.786200
glass        CF            3.0      0.701883        0.739203
             ENN           3.0      0.701883        0.710321
             ENNTh         3.0      0.701883        0.698007
             NCNEdit       3.0      0.701883        0.746777
heart        CF            3.0      0.740000        0.837037
             ENN           3.0      0.740000        0.847407
             ENNTh         3.0      0.740000        0.822222
             NCNEdit       3.0      0.740000        0.860741
ionosphere   CF            3.0      0.844467        0.861062
             ENN           3.0      0.844467        0.846785
             ENNTh         3.0      0.844467        0.820604
             NCNEdit       3.0      0.844467        0.874157
iris         CF            3.0      0.922667        0.954667
             ENN           3.0      0.922667        0.953333
             ENNTh         3.0      0.922667        0.946667
             NCNEdit       3.0      0.922667        0.957333
lymphography CF            3.0      0.768414        0.828138
             ENN           3.0      0.768414        0.843172
             ENNTh         3.0      0.768414        0.798345
             NCNEdit       3.0      0.768414        0.856414
monk-2       CF            3.0      0.714301        0.830580
             ENN           3.0      0.714301        0.859262
             ENNTh         3.0      0.714301        0.830997
             NCNEdit       3.0      0.714301        0.863908
newthyroid   CF            3.0      0.929302        0.958140
             ENN           3.0      0.929302        0.947907